# Q3 — 9칸 복수 위치 평가 v1 (기존 정답 보존)

**준비·CPU 검증 완료, 실제 LVLM 효과 검증은 아직 시작 전.**

이 파일은 실행 코드를 내부에 포함하므로 GitHub push/토큰 없이 열 수 있습니다.
처음에는 CPU 런타임에서 `RUN_INFERENCE=False` 그대로 **모두 실행**하세요. Drive 연결 후 평가 입력만 준비하며 모델 다운로드·추론은 없습니다.
실제 추론은 나중에 L4/A100 런타임에서 `RUN_INFERENCE=True`로 명시적으로 바꿨을 때만 실행합니다.

- 정상107 / 위치정답 결함133 / 위치미상 결함6 = 전체246장.
- 떨어진 결함은 박스마다 중심 칸을 따로 정답에 포함합니다. 넓은 박스의 전체 면적이나 모든 결함 개수를 평가하는 방식은 아닙니다.
- 위치미상6장은 유무 평가에는 포함하고 위치 점수에서만 제외합니다.
- direct와 유무→위치 방식의 호출 수는 다르므로 구조 자체의 인과 효과는 주장하지 않습니다.
- 기존 gold·133개 4지선다·수동 라벨을 덮어쓰지 않습니다. gold는 이미 탐색된 자료이며 dev 방법 선택을 대체하지 않습니다.
- 녹 사진 `01-06-2021_DJI_0335_100.jpg`, `01-06-2021_DJI_0547_159.jpg`는 박스 밖 표면 변화도 결함인지 추가 검토가 남아 있습니다. 이 파일은 라벨을 임의 수정하지 않습니다.

In [ ]:
RUN_INFERENCE = False
QA_FOLDER = "lvlm_insplad_qa/gold_v2_final"
LOC_FOLDER = "lvlm_insplad_qa/gold_v2_loc_mmad"
RUN_FOLDER = "lvlm_insplad_results/gold_v2_q3_multiregion_v1"
POLICIES = ["direct", "staged"]
SUP_ZIP_PATH = ""     # 비우면 Drive에서 공식 ZIP 이름/크기로 검색
UNSUP_ZIP_PATH = ""

In [ ]:
import base64, hashlib, io, json, sys, zipfile
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
RUN_ROOT = DRIVE_ROOT / RUN_FOLDER
if not RUN_ROOT.name.startswith("gold_v2_q3_multiregion"):
    raise ValueError("별도 gold_v2_q3_multiregion 출력 폴더를 사용하세요.")
SOURCE_SHA256 = '1244b1fd6f6536e15ecf6be4106582304a2b70c751acee0bd66d37f37059f7dd'
SOURCE_FILES = {'scripts/q3_multiregion.py': 'c431048fec6e06e6f26e2f57628e74a7861765aa4f0e63905ae16c3b40d12752', 'scripts/run_q3_multiregion.py': '33499d274b45c96b9a24d49b9c50d42a45df6edba1df9e5ba817f7c846749724', 'scripts/run_gold_baselines.py': '978cb258c7412a767de2b5f7ee9604e726c29dc41922d656084d751d32869575', 'scripts/build_explainable_qa_gold.py': '9acdd42e87c15f9da2ba9bb7c3f89d9e77c42d774a8a8d38f6e783ea544d4f0a', 'src/insplad_benchmark/__init__.py': '5d9678c34324691fa8d7d5b507501945bb6eafd32e53ef6dedf1979e005cca49', 'src/insplad_benchmark/hf_adapters.py': 'dbf178f1a6263c371fc8c7db192a33bdd252910b63267583d5bc24db11f684ab'}
SOURCE_BYTES = base64.b64decode('...')
assert hashlib.sha256(SOURCE_BYTES).hexdigest() == SOURCE_SHA256
CODE_DIR = Path("/content/q3_multiregion_" + SOURCE_SHA256[:12])
with zipfile.ZipFile(io.BytesIO(SOURCE_BYTES)) as z:
    assert set(z.namelist()) == set(SOURCE_FILES)
    for name in z.namelist():
        target = CODE_DIR / name
        assert target.resolve().is_relative_to(CODE_DIR.resolve())
        data = z.read(name)
        assert hashlib.sha256(data).hexdigest() == SOURCE_FILES[name]
        if target.exists() and target.read_bytes() != data:
            raise RuntimeError("내장 소스와 런타임 파일 불일치. 새 런타임을 사용하세요.")
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
sys.path[:0] = [str(CODE_DIR / "scripts"), str(CODE_DIR / "src")]
import q3_multiregion as q3
print("소스 고정:", SOURCE_SHA256)

In [ ]:
import subprocess
EXPECTED_INPUT_SHA256 = {'master': '467beb9bb29223ca84c877995b632c682039f1729bebb239e7f9ee6691ed2d27', 'location': 'f5a6b5c1e0b9d7061da729b4b5186882e10dca00fb94a4de998b798e9cf1e5a5', 'annotations': 'cd1592f2a83f409d4e81bab16f049e9fec903cb7cf7db1c76edd6bf33b0044c1'}
QA_DIR = DRIVE_ROOT / QA_FOLDER
LOC_DIR = DRIVE_ROOT / LOC_FOLDER
MASTER_QA = QA_DIR / "master_qa_492.jsonl"
LOC_QA = LOC_DIR / "localization_mmad_qa_133.jsonl"
ANNOTATIONS = LOC_DIR / "annotations.json"
for key, path in [("master", MASTER_QA), ("location", LOC_QA), ("annotations", ANNOTATIONS)]:
    if not path.is_file():
        raise FileNotFoundError(path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != EXPECTED_INPUT_SHA256[key]:
        raise RuntimeError("검증한 원본과 다른 입력: " + str(path) + ". 이 파일은 덮어쓰지 않습니다.")
INPUT_DIR = RUN_ROOT / "inputs"
subprocess.run([sys.executable, str(CODE_DIR / "scripts/q3_multiregion.py"),
    "--master-qa", str(MASTER_QA), "--loc-qa", str(LOC_QA),
    "--annotations", str(ANNOTATIONS), "--out-dir", str(INPUT_DIR)], check=True)
DATASET = INPUT_DIR / "dataset.json"
rows = json.loads(DATASET.read_text(encoding="utf-8"))
assert len(rows) == 246
assert sum(r["anomaly_gt"] == "normal" for r in rows) == 107
assert sum(r["regions_gt"] is None for r in rows) == 6
assert sum(bool(r["regions_gt"]) for r in rows) == 133
print("입력 준비 완료. 246장, 위치 정답133장. Q1/Q2 호출0.")
print("항상없음/항상중앙/항상9칸 대조군:")
print((INPUT_DIR / "constant_baselines.json").read_text(encoding="utf-8"))

## 실제 추론 — 명시적으로 켰을 때만

`RUN_INFERENCE=True`와 VRAM20GB 이상 GPU가 필요합니다. 직접 위치 질문246회 + 유무 질문246회 + 유무 양성일 때만 위치 질문(최대246회), 합계492~738회입니다. Q1/Q2 신규 호출은 없습니다.
모델은 OpenGVLab/InternVL3_5-8B의 첫 실행 revision을 Drive에 고정하고 이후 같은 revision만 재사용합니다. 모델 다운로드에는 저장공간과 시간이 필요합니다.
각 호출 직후 저장합니다. 끊기면 같은 설정으로 다시 실행하고, 오류 호출만 재시도합니다. invalid 응답을 정답이 나올 때까지 다시 묻지 않습니다.

In [ ]:
INFERENCE_DIR = RUN_ROOT / "inference_v1"
if not RUN_INFERENCE:
    print("준비만 완료했습니다. 모델 다운로드/추론0회. 나중에 L4/A100에서 RUN_INFERENCE=True로 실행하세요.")
else:
    import torch
    if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20 * 1024**3:
        raise RuntimeError("VRAM20GB 이상 L4/A100 GPU가 필요합니다.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.14.0",
        "accelerate==1.14.0", "sentencepiece==0.2.2", "numpy>=1.26", "pillow>=10", "timm"], check=True)
    from huggingface_hub import HfApi, snapshot_download
    receipt_path = RUN_ROOT / "model_revision.json"
    model_id = "OpenGVLab/InternVL3_5-8B"
    if receipt_path.exists():
        receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
        assert receipt["model_id"] == model_id
        revision = receipt["revision"]
    else:
        revision = HfApi().model_info(model_id).sha
        q3.atomic_json(receipt_path, {"model_id": model_id, "revision": revision})
    snapshot = snapshot_download(model_id, revision=revision)
    def find_zip(filename, size, override):
        candidates = [Path(override)] if override else []
        candidates += [DRIVE_ROOT / filename, DRIVE_ROOT / "lvlm_insplad_data" / filename]
        for path in candidates:
            if path.is_file() and path.stat().st_size == size:
                return path
        for path in DRIVE_ROOT.rglob(filename):
            if path.is_file() and path.stat().st_size == size:
                return path
        raise FileNotFoundError(filename)
    sup = find_zip("supervised_fault_classification.zip", 874753167, SUP_ZIP_PATH)
    uns = find_zip("unsupervised_anomaly_detection.zip", 1170967719, UNSUP_ZIP_PATH)
    subprocess.run([sys.executable, str(CODE_DIR / "scripts/run_q3_multiregion.py"),
        "--dataset", str(DATASET), "--sup-zip", str(sup), "--unsup-zip", str(uns),
        "--model-snapshot", snapshot, "--out-dir", str(INFERENCE_DIR),
        "--policies", *POLICIES], check=True)

In [ ]:
if RUN_INFERENCE:
    status = json.loads((INFERENCE_DIR / "run_status.json").read_text(encoding="utf-8"))
    print("완료 상태:", status)
    if status["status"] != "complete" or status.get("fixture"):
        raise RuntimeError("실제 추론이 완료되지 않았으므로 결과를 내보내지 않습니다.")
    for policy in POLICIES:
        result = json.loads((INFERENCE_DIR / f"metrics_{policy}.json").read_text(encoding="utf-8"))
        print(policy, json.dumps(result["overall"], ensure_ascii=False, indent=2))
    print("invalid가 있으면 invalid 비율과 함께 점수를 해석하세요. 오류 호출은 재실행으로 회수하세요.")
    export_path = RUN_ROOT / "q3_multiregion_results.zip"
    with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(INFERENCE_DIR.glob("*.json")):
            z.write(path, "run/" + path.name)
        for path in sorted(INPUT_DIR.glob("*.json")):
            z.write(path, "inputs/" + path.name)
        z.write(receipt_path, "model_revision.json")
        z.writestr("source_bundle.zip", SOURCE_BYTES)
    print("ZIP:", export_path, "SHA256:", hashlib.sha256(export_path.read_bytes()).hexdigest())
    from google.colab import files
    files.download(str(export_path))
else:
    print("평가 실행 전입니다. 입력 준비만 완료했습니다:", INPUT_DIR)